# Sphere Model - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC simulation results for a simple sphere geometry using the ENDF/B-VIII.1 nuclear data library.

## Model Description

- **Geometry**: 35 cm radius sphere with vacuum boundary
- **Source**: Point source at origin, 14.06 MeV monoenergetic neutrons, isotropic
- **Material**: Single nuclide at 1.0 g/cm³, temperature 294 K
- **Statistics**: 500,000 particles × 10 batches = 5,000,000 total histories
- **Tallies**: Flux, heating, and all available reaction MTs (energy-integrated and energy-binned)

In [1]:
# Setup and imports
import numpy as np
import pandas as pd
import glob
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
PARTICLES = 500000
BATCHES = 10
SOURCE_ENERGY = 14.06e6  # eV
MAX_PLOT_ENERGY = SOURCE_ENERGY * 1.1  # Source energy + 10%
FLUX_DECADES = 6  # Number of decades to show on flux y-axis (from max value)

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None
# Compute backend(s) for the YAMC runs. Mirrors the env-var pattern from
# tasks/cpu-vs-gpu/sphere.ipynb so the same convention drives both notebooks.
#   YAMC_VERIFY_COMPUTE_MODE=cpu   (default) — CPU only
#   YAMC_VERIFY_COMPUTE_MODE=gpu             — GPU only
#   YAMC_VERIFY_COMPUTE_MODE=both            — CPU and GPU side-by-side
_VALID_MODES = {"cpu", "gpu", "both"}
COMPUTE_MODE = os.environ.get("YAMC_VERIFY_COMPUTE_MODE", "cpu").strip().lower()
if COMPUTE_MODE not in _VALID_MODES:
    raise ValueError(
        f"YAMC_VERIFY_COMPUTE_MODE must be one of {sorted(_VALID_MODES)}; "
        f"got {COMPUTE_MODE!r}"
    )
COMPUTES = ["cpu", "gpu"] if COMPUTE_MODE == "both" else [COMPUTE_MODE]
# No precheck for GPU availability — the dispatch raises a clean
# RuntimeError if no f64 Vulkan adapter is present, which is enough
# signal for both CI and interactive use.
print(f"Compute backend(s): {COMPUTES}")


Compute backend(s): ['cpu', 'gpu']


In [2]:
def run_openmc_simulation(nuclide, nuclear_data):
    """Run OpenMC simulation for a single nuclide."""
    import openmc as mc
    
    mc.config['cross_sections'] = f"{NUCLEAR_DATA_DIR[nuclear_data]}cross_sections.xml"
    
    # Geometry
    sphere1 = mc.Sphere(
        surface_id=2,
        x0=0.0, y0=0.0, z0=0.0,
        r=35.0,
        boundary_type='vacuum',
    )
    region1 = -sphere1
    
    # Material
    material1 = mc.Material()
    material1.material_id = 1
    material1.add_nuclide(nuclide, 1.0)
    material1.set_density("g/cm3", 1.0)
    material1.temperature = 294
    
    # Cell
    cell1 = mc.Cell(
        cell_id=1,
        name="inner_sphere",
        region=region1,
        fill=material1
    )
    geometry = mc.Geometry([cell1])
    
    # Get available MTs
    in_neu = mc.data.IncidentNeutron.from_hdf5(f"{NUCLEAR_DATA_DIR[nuclear_data]}neutron/{nuclide}.h5")
    mts_int = [mt for mt in in_neu.reactions.keys() if mt not in (1, 3, 4, 27, 101)]
    mts = [str(mt) for mt in mts_int]
    
    # Filters and tallies
    cell_filter = mc.CellFilter(cell1)
    energy_filter = mc.EnergyFilter.from_group_structure('VITAMIN-J-175')
    
    # Get energy bin boundaries - OpenMC returns (n_bins, 2) array with [low, high]
    energy_bins_2d = energy_filter.bins
    energy_bins = np.concatenate([[energy_bins_2d[0, 0]], energy_bins_2d[:, 1]])
    
    tallies = []
    
    all_scores = mts + ['total', 'absorption', 'flux', 'heating', 'heating-local', 'H1-production', 'H2-production', 'H3-production', 'He3-production', 'He4-production']
    for score in all_scores:                                                                                                                                
        tally = mc.Tally()                                                                                                                                  
        tally.filters = [cell_filter]                                                                                                                       
        tally.scores = [score]                                                                                                                              
        tally.name = str(score)                                                                                                                             
        tallies.append(tally)    
    
    # Energy-binned flux tally
    tally_spectrum = mc.Tally()
    tally_spectrum.filters = [cell_filter, energy_filter]
    tally_spectrum.scores = ['flux']
    tally_spectrum.name = "flux_energy_binned"
    tallies.append(tally_spectrum)
    
    # Source and settings
    source = mc.IndependentSource(
        angle=mc.stats.Isotropic(),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        space=mc.stats.Point([0, 0, 0])
    )
    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False}
    )
    
    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    sp_filename = model.run(apply_tally_results=True)
    sp = mc.StatePoint(sp_filename)
    
    # Collect results
    transport_time = sp.runtime['transport']
    total_particles = PARTICLES * BATCHES
    particles_per_second = total_particles / transport_time
    
    results = {
        'nuclide': nuclide,
        'code': 'openmc',
        'particles/s': particles_per_second,
        'tallies': {},
        'tally_std': {}
    }
    
    for tally in tallies:
        if tally.name != "flux_energy_binned":
            results['tallies'][tally.name] = tally.mean.flatten()[0]
            results['tally_std'][tally.name] = tally.std_dev.flatten()[0]
    
    # Get spectrum tally
    results['flux_spectra_mean'] = np.array(tally_spectrum.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(tally_spectrum.std_dev).squeeze()
    results['energy_bins'] = energy_bins
    
    return results

In [3]:
def run_yamc_simulation(nuclide, nuclear_data, compute="cpu"):
    """Run YAMC simulation for a single nuclide. Returns (results_dict, model)."""
    import yamc as mc

    sphere1 = mc.Sphere(
        surface_id=2,
        x0=0.0, y0=0.0, z0=0.0,
        radius=35.0,
        boundary='vacuum',
    )
    region1 = -sphere1

    material1 = mc.Material(
        composition={nuclide: 1.0},
        density=1.0,
        temperature=294,
        id=1,
    )
    # Keyword library — yamc autodownloads Arrow data into ~/.cache/yamc on first use
    material1.read_nuclear_data('endf-b8.1')

    mts = [mt for mt in material1.reaction_mts if mt not in (1, 3, 4, 27, 101)]

    cell1 = mc.Cell(
        id=1,
        name="inner_sphere",
        region=region1,
        material=material1,
    )
    geometry = mc.Geometry([cell1])

    tallies = []
    all_scores = mts + ['total', 'absorption', 'flux', 'heating', 'heating-local', 'H1-production', 'H2-production', 'H3-production', 'He3-production', 'He4-production']
    for score in all_scores:
        tally = mc.Tally(
            scores=[str(score)],
            name=str(score),
            cells=cell1,
            particle="neutron",
        )
        tallies.append(tally)

    tally_spectrum = mc.Tally(
        scores=['flux'],
        name="flux_energy_binned",
        cells=cell1,
        particle="neutron",
        energy_group_structure='VITAMIN-J-175',
    )
    tallies.append(tally_spectrum)
    energy_bins = np.array(tally_spectrum.energy_bins)

    source = mc.NeutronSource(
        position=mc.Point(0, 0, 0),
        direction=mc.Isotropic(),
        energy=mc.Discrete([SOURCE_ENERGY], [1.0]),
    )

    model = mc.Model(
        geometry=geometry,
        tallies=tallies,
        source=source,
        particles_per_batch=PARTICLES,
        batches=BATCHES,
        seed=1,
    )
    import time as _t
    _t0 = _t.perf_counter()
    sim_results = model.simulate_transport(compute=compute)
    _elapsed = _t.perf_counter() - _t0

    results = {
        'nuclide': nuclide,
        'code': f'yamc-{compute}',
        'particles/s': (
            model.particles_per_second
            if model.particles_per_second is not None
            else (PARTICLES * BATCHES) / _elapsed if _elapsed > 0 else 0
        ),
        'tallies': {},
        'tally_std': {}
    }
    for tally in tallies:
        if tally.name != "flux_energy_binned":
            r = sim_results[tally]
            results['tallies'][tally.name] = r.mean[0]
            results['tally_std'][tally.name] = r.standard_deviation[0]

    spec = sim_results[tally_spectrum]
    results['flux_spectra_mean'] = np.array(spec.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(spec.standard_deviation).squeeze()
    results['energy_bins'] = energy_bins

    return results, model

In [4]:
def create_step_trace(energy_bins, values, name, color, dash=None, showlegend=True):
    """Create a step plot trace for Plotly (similar to matplotlib step with where='post')."""
    # For step plot: duplicate x values and create staircase y values
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])

    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
        showlegend=showlegend,
    )


# Per-backend plot styling. Keys are backend labels used in the legend.
BACKEND_STYLES = {
    'OpenMC': dict(color='blue', dash=None, marker='circle'),
    'YAMC (cpu)': dict(color='red', dash='dash', marker='square'),
    'YAMC (gpu)': dict(color='orange', dash='dot', marker='diamond'),
}
RATIO_COLORS = {
    'YAMC (cpu) / OpenMC': 'green',
    'YAMC (gpu) / OpenMC': 'purple',
}


def create_comparison_plot(openmc_results, yamc_results_by_backend):
    """Create a Plotly subplot with all spectra overlaid (top) and per-backend ratios (bottom).

    Parameters
    ----------
    openmc_results : dict
        OpenMC results dictionary.
    yamc_results_by_backend : dict[str, dict]
        Maps backend name (e.g. "cpu", "gpu") to a YAMC results dictionary.
    """

    energy_bins = openmc_results['energy_bins']
    # Geometric mean for bin centers (better for log scale)
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])

    # Lethargy normalization (shared across all spectra)
    lethargy_bin_width = np.log10(energy_bins[1:] / energy_bins[:-1])

    openmc_flux = openmc_results['flux_spectra_mean'].copy() / lethargy_bin_width
    openmc_std = openmc_results['flux_spectra_std_dev'].copy() / lethargy_bin_width

    # Normalize each YAMC backend
    yamc_flux_by_backend = {}
    yamc_std_by_backend = {}
    for backend, res in yamc_results_by_backend.items():
        yamc_flux_by_backend[backend] = res['flux_spectra_mean'].copy() / lethargy_bin_width
        yamc_std_by_backend[backend] = res['flux_spectra_std_dev'].copy() / lethargy_bin_width

    # Find the lowest energy bin with non-zero flux in any code
    has_flux = openmc_flux > 0
    for f in yamc_flux_by_backend.values():
        has_flux = has_flux | (f > 0)
    if np.any(has_flux):
        first_nonzero_idx = np.argmax(has_flux)
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]

    # Calculate flux threshold from the global maximum across all spectra
    all_positive_max = [np.max(openmc_flux[openmc_flux > 0])] if np.any(openmc_flux > 0) else []
    for f in yamc_flux_by_backend.values():
        if np.any(f > 0):
            all_positive_max.append(np.max(f[f > 0]))
    max_flux = max(all_positive_max) if all_positive_max else 1e-10
    flux_threshold = max_flux / (10 ** FLUX_DECADES)

    def _mask(values, std, ref):
        keep = ref >= flux_threshold
        return np.where(keep, values, np.nan), np.where(keep, std, np.nan)

    openmc_flux_plot, openmc_std_plot = _mask(openmc_flux, openmc_std, openmc_flux)
    yamc_plot_by_backend = {}
    for backend in yamc_results_by_backend:
        f = yamc_flux_by_backend[backend]
        s = yamc_std_by_backend[backend]
        yamc_plot_by_backend[backend] = _mask(f, s, f)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=('Flux per Unit Lethargy', 'YAMC / OpenMC Ratio')
    )

    # --- Top plot: all spectra overlaid ---
    openmc_style = BACKEND_STYLES['OpenMC']
    fig.add_trace(
        create_step_trace(energy_bins, openmc_flux_plot, 'OpenMC',
                          openmc_style['color'], dash=openmc_style['dash']),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=bin_centers,
            y=openmc_flux_plot,
            mode='markers',
            name='OpenMC error',
            marker=dict(color=openmc_style['color'], size=4, symbol=openmc_style['marker']),
            error_y=dict(type='data', array=openmc_std_plot, visible=True,
                         color=openmc_style['color'], thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )

    for backend, (flux_plot, std_plot) in yamc_plot_by_backend.items():
        label = f'YAMC ({backend})'
        style = BACKEND_STYLES.get(label, dict(color='red', dash='dash', marker='square'))
        fig.add_trace(
            create_step_trace(energy_bins, flux_plot, label,
                              style['color'], dash=style['dash']),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=bin_centers,
                y=flux_plot,
                mode='markers',
                name=f'{label} error',
                marker=dict(color=style['color'], size=4, symbol=style['marker']),
                error_y=dict(type='data', array=std_plot, visible=True,
                             color=style['color'], thickness=1),
                showlegend=False,
            ),
            row=1, col=1
        )

    # --- Bottom plot: ratio per backend ---
    for backend, res in yamc_results_by_backend.items():
        yamc_flux = yamc_flux_by_backend[backend]
        yamc_std = yamc_std_by_backend[backend]
        valid = (openmc_flux > 0) & (yamc_flux > 0)
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(valid, yamc_flux / openmc_flux, np.nan)
            rel_err_yamc = np.where(yamc_flux > 0, yamc_std / yamc_flux, 0)
            rel_err_openmc = np.where(openmc_flux > 0, openmc_std / openmc_flux, 0)
        ratio_rel_err = np.sqrt(rel_err_yamc**2 + rel_err_openmc**2)
        ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)

        ratio_label = f'YAMC ({backend}) / OpenMC'
        ratio_color = RATIO_COLORS.get(ratio_label, 'green')
        # Convert color name to rgba for fill transparency
        rgba_fill = {
            'green': 'rgba(0, 128, 0, 0.2)',
            'purple': 'rgba(128, 0, 128, 0.2)',
        }.get(ratio_color, 'rgba(0, 128, 0, 0.2)')

        fig.add_trace(
            create_step_trace(energy_bins, ratio, ratio_label, ratio_color),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=np.concatenate([bin_centers, bin_centers[::-1]]),
                y=np.concatenate([ratio + ratio_std, (ratio - ratio_std)[::-1]]),
                fill='toself',
                fillcolor=rgba_fill,
                line=dict(color='rgba(255,255,255,0)'),
                name=f'{ratio_label} uncertainty',
                showlegend=False,
            ),
            row=2, col=1
        )

    # Reference line at 1.0
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

    # Update axes with scientific notation format and x-axis limits
    fig.update_xaxes(
        type="log",
        title_text="Energy (eV)",
        range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
        exponentformat='e',
        row=2, col=1
    )
    fig.update_xaxes(
        type="log",
        range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
        exponentformat='e',
        row=1, col=1
    )
    fig.update_yaxes(
        type="log",
        title_text="Flux / Lethargy",
        exponentformat='e',
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Ratio",
        range=[0.5, 1.5],
        row=2, col=1
    )

    fig.update_layout(
        height=700,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=60, r=20, t=40, b=40)
    )

    return fig


def calculate_spectral_agreement(openmc_results, yamc_results, flux_threshold=1e-6):
    """Calculate statistical agreement metrics for flux spectra comparison."""
    
    openmc_flux = openmc_results['flux_spectra_mean']
    openmc_std = openmc_results['flux_spectra_std_dev']
    yamc_flux = yamc_results['flux_spectra_mean']
    yamc_std = yamc_results['flux_spectra_std_dev']
    
    chi_squared_terms = []
    rel_diffs = []
    weighted_rel_diffs = []
    flux_weights = []
    
    for i in range(len(openmc_flux)):
        omc = openmc_flux[i]
        ymc = yamc_flux[i]
        omc_std = openmc_std[i]
        ymc_std = yamc_std[i]
        
        # Chi-squared term: (observed - expected)^2 / (sigma_obs^2 + sigma_exp^2)
        combined_std = np.sqrt(omc_std**2 + ymc_std**2)
        if combined_std > 0:
            chi_squared_terms.append(((ymc - omc) / combined_std)**2)
        
        # Relative difference (only for bins with significant flux)
        if omc > flux_threshold or ymc > flux_threshold:
            if omc > 0:
                rel_diff = abs((ymc - omc) / omc) * 100
                if np.isfinite(rel_diff):
                    rel_diffs.append(rel_diff)
                    weight = max(omc, ymc)
                    weighted_rel_diffs.append(rel_diff * weight)
                    flux_weights.append(weight)
    
    # Calculate metrics
    chi_squared = np.sum(chi_squared_terms)
    dof = len(chi_squared_terms)
    reduced_chi_squared = chi_squared / dof if dof > 0 else 0.0
    
    median_rel_diff = np.median(rel_diffs) if rel_diffs else 0.0
    weighted_avg_rel_diff = np.sum(weighted_rel_diffs) / np.sum(flux_weights) if flux_weights else 0.0
    
    # Quality assessment
    if reduced_chi_squared < 1.5:
        quality = "Excellent"
    elif reduced_chi_squared < 3.0:
        quality = "Good"
    elif reduced_chi_squared < 5.0:
        quality = "Fair"
    else:
        quality = "Poor"
    
    return {
        'reduced_chi_squared': reduced_chi_squared,
        'degrees_of_freedom': dof,
        'median_rel_diff': median_rel_diff,
        'weighted_avg_rel_diff': weighted_avg_rel_diff,
        'quality': quality,
        'bins_analyzed': len(rel_diffs),
        'total_bins': len(openmc_flux)
    }


def create_agreement_table(stats):
    """Create a summary table of spectral agreement statistics."""
    rows = [
        {'Metric': 'Reduced Chi-Squared', 'Value': f"{stats['reduced_chi_squared']:.2f}", 
         'Note': f"(ideal ~ 1.0, dof={stats['degrees_of_freedom']})"},
        {'Metric': 'Weighted Avg Rel. Diff', 'Value': f"{stats['weighted_avg_rel_diff']:.2f}%",
         'Note': '(flux-weighted)'},
        {'Metric': 'Median Rel. Diff', 'Value': f"{stats['median_rel_diff']:.2f}%",
         'Note': '(robust to outliers)'},
        {'Metric': 'Agreement Quality', 'Value': stats['quality'],
         'Note': f"({stats['bins_analyzed']}/{stats['total_bins']} bins analyzed)"},
    ]
    return pd.DataFrame(rows)


# MT number to reaction description mapping from YAMC
import yamc
MT_DESCRIPTIONS = yamc.reaction_names()

# Score name to MT number mapping (reverse lookup + special score aliases)
SCORE_TO_MT = {name: mt for mt, name in MT_DESCRIPTIONS.items()}
SCORE_TO_MT.update({
    'total': 1,
    'elastic': 2,
    'absorption': 27,
    'fission': 18,
    'heating': 301,
    'heating-local': 901,
    'H1-production': 203,
    'H2-production': 204,
    'H3-production': 205,
    'He3-production': 206,
    'He4-production': 207,
})


def get_mt_and_reaction(score_name):
    """Get MT number and reaction description for a score.
    
    Returns (mt_display, reaction_description) tuple.
    """
    # Try to parse as integer MT number
    try:
        mt = int(score_name)
        return str(mt), MT_DESCRIPTIONS.get(mt, f'MT {mt}')
    except ValueError:
        pass
    
    # Check if it's a known score name with an MT equivalent
    if score_name in SCORE_TO_MT:
        mt = SCORE_TO_MT[score_name]
        return str(mt), score_name
    
    # Unknown non-numeric score (flux, production scores, etc.)
    return '-', score_name


def _best_value(results, score_names):
    """Pick the best (non-nan, non-zero preferred) tally value across alias score names."""
    best_val = np.nan
    best_std = np.nan
    for name in score_names:
        val = results['tallies'].get(name, np.nan)
        std = results['tally_std'].get(name, np.nan)
        if not np.isnan(val) and val != 0:
            if np.isnan(best_val) or best_val == 0:
                best_val = val
                best_std = std
        elif np.isnan(best_val):
            best_val = val
            best_std = std
    return best_val, best_std


def _is_zero_or_nan(value):
    return value == 0 or np.isnan(value)


def _format_tally(value, std):
    if np.isnan(value):
        return 'N/A'
    return f"{value:.4e} +/- {std:.2e}"


def _format_rel_diff(yamc_val, openmc_val):
    if np.isnan(openmc_val) or openmc_val == 0 or np.isnan(yamc_val):
        return 'N/A'
    return f"{(yamc_val - openmc_val) / openmc_val * 100:.2f}%"


def create_tally_table(openmc_results, yamc_results_by_backend):
    """Build a comparison table with one column per code/backend (OpenMC, YAMC cpu, YAMC gpu).

    Parameters
    ----------
    openmc_results : dict
        OpenMC results dictionary.
    yamc_results_by_backend : dict[str, dict]
        Maps backend name (e.g. "cpu", "gpu") to a YAMC results dictionary.
    """

    # Collect all tally names across every results dict
    all_tallies = set(openmc_results['tallies'].keys())
    for res in yamc_results_by_backend.values():
        all_tallies |= set(res['tallies'].keys())

    # Group tallies by MT number to deduplicate (e.g., "1" and "total" both map to MT 1)
    mt_to_scores = {}
    for name in all_tallies:
        mt_display, _ = get_mt_and_reaction(name)
        mt_to_scores.setdefault(mt_display, []).append(name)

    rows = []
    for mt_display, score_names in mt_to_scores.items():
        openmc_val, openmc_std = _best_value(openmc_results, score_names)
        yamc_vals = {}
        for backend, res in yamc_results_by_backend.items():
            yamc_vals[backend] = _best_value(res, score_names)

        # Skip rows where every code reports zero (or nan)
        all_zero = _is_zero_or_nan(openmc_val) and all(
            _is_zero_or_nan(v[0]) for v in yamc_vals.values()
        )
        if all_zero:
            continue

        # Canonical reaction name from MT_DESCRIPTIONS when available
        reaction = None
        if mt_display != '-':
            try:
                reaction = MT_DESCRIPTIONS.get(int(mt_display))
            except ValueError:
                pass
        if reaction is None:
            _, reaction = get_mt_and_reaction(score_names[0])

        row = {
            'MT': mt_display,
            'Reaction': reaction,
            'OpenMC (cpu)': _format_tally(openmc_val, openmc_std),
        }
        for backend, (val, std) in yamc_vals.items():
            row[f'YAMC ({backend})'] = _format_tally(val, std)
        for backend, (val, _std) in yamc_vals.items():
            row[f'Rel. Diff {backend} (%)'] = _format_rel_diff(val, openmc_val)

        row['_mt_sort'] = int(mt_display) if mt_display != '-' else 9999
        rows.append(row)

    rows.sort(key=lambda r: r['_mt_sort'])
    for row in rows:
        del row['_mt_sort']

    return pd.DataFrame(rows)

In [5]:
# Get list of nuclides
h5_files = sorted(glob.glob(os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron', "*.h5")))
nuclides = [os.path.splitext(os.path.basename(f))[0] for f in h5_files]

if not nuclides:
    raise FileNotFoundError(
        f"No HDF5 files found in {NUCLEAR_DATA_DIR[NUCLEAR_DATA]}neutron/\n"
        f"Please check that NUCLEAR_DATA_DIR['{NUCLEAR_DATA}'] points to a valid nuclear data directory."
    )

# Skip metastable nuclides for transport
nuclides = [n for n in nuclides if "_" not in n]

if NUMBER_OF_SIMULATIONS:
    nuclides = nuclides[:NUMBER_OF_SIMULATIONS]

print(f"Running simulations for {len(nuclides)} nuclides: {nuclides}")

Running simulations for 2 nuclides: ['Ac225', 'Ac226']


In [6]:
# Run all simulations and collect results
all_results = {}
first_yamc_model = None

for nuclide in nuclides:
    print(f"\nProcessing {nuclide}...")
    try:
        openmc_results = run_openmc_simulation(nuclide, NUCLEAR_DATA)
        per_nuclide = {'openmc': openmc_results}

        for compute in COMPUTES:
            yamc_results, yamc_model = run_yamc_simulation(
                nuclide, NUCLEAR_DATA, compute=compute,
            )
            per_nuclide[f'yamc-{compute}'] = yamc_results
            if first_yamc_model is None:
                first_yamc_model = yamc_model

        all_results[nuclide] = per_nuclide
        print(f"  OpenMC: {openmc_results['particles/s']:.0f} particles/s")
        for compute in COMPUTES:
            yamc_pps = per_nuclide[f'yamc-{compute}']['particles/s']
            print(f"  YAMC ({compute}): {yamc_pps:.0f} particles/s")
    except Exception as e:
        print(f"  Error: {e}")
        continue

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")



Processing Ac225...


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

 Simulating batch 2


 Simulating batch 3


 Simulating batch 4


 Simulating batch 5


... (35 outputs left out)

Starting batch 9/10


Starting batch 10/10


Transport simulation complete.
Time: 2.535 s (data load: 0.000 s, transport: 2.535 s) | Particles/s: 1,972,225 (transport only: 1,972,471) | Total: 5,000,000 particles


  OpenMC: 1568387 particles/s
  YAMC (cpu): 1972225 particles/s
  YAMC (gpu): 4806631 particles/s

Completed 2 / 2 nuclides successfully.


In [7]:
# Display sphere model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", n_samples=500, axis_units="cm")
    html_plot.save("sphere_model_plot.html")
    encoded = base64.b64encode(html_plot.html.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

/home/jon/yamc-org/yamc/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- Interactive flux spectra plot (log-log scale)
- YAMC/OpenMC ratio plot with uncertainty band
- Tally comparison table sorted by relative difference
- Performance comparison (particles/second)

In [8]:
# Display results for each nuclide
for nuclide, results in all_results.items():
    openmc_results = results['openmc']

    # Collect available YAMC backends in the order the user requested
    yamc_results_by_backend = {}
    for compute in COMPUTES:
        yres = results.get(f'yamc-{compute}')
        if yres is not None:
            yamc_results_by_backend[compute] = yres

    display(Markdown(f"## {nuclide}"))

    # Performance line — one entry per code/backend
    perf_parts = [f"OpenMC: {openmc_results['particles/s']:,.0f} particles/s"]
    for compute, yres in yamc_results_by_backend.items():
        perf_parts.append(f"YAMC ({compute}): {yres['particles/s']:,.0f} particles/s")
    display(Markdown(f"**Performance:** " + " | ".join(perf_parts)))

    # Combined flux spectra plot (OpenMC + every YAMC backend)
    fig = create_comparison_plot(openmc_results, yamc_results_by_backend)
    fig.update_layout(title=f"{nuclide} - Flux Spectra Comparison")
    fig.show()

    # Spectral agreement statistics: one block per backend
    for compute, yres in yamc_results_by_backend.items():
        heading = (
            f"#### Spectral Agreement (YAMC {compute} vs OpenMC)"
            if len(yamc_results_by_backend) > 1
            else "#### Spectral Agreement"
        )
        display(Markdown(heading))
        stats = calculate_spectral_agreement(openmc_results, yres)
        display(create_agreement_table(stats).style.hide(axis='index'))

    # Combined tally table with OpenMC + every YAMC backend column
    display(Markdown("#### Tally Comparison"))
    table = create_tally_table(openmc_results, yamc_results_by_backend)
    display(table.style.hide(axis='index'))

    print()  # spacing


## Ac225

**Performance:** OpenMC: 1,471,832 particles/s | YAMC (cpu): 1,858,423 particles/s | YAMC (gpu): 2,957,069 particles/s

#### Spectral Agreement (YAMC cpu vs OpenMC)

Metric,Value,Note
Reduced Chi-Squared,0.88,"(ideal ~ 1.0, dof=146)"
Weighted Avg Rel. Diff,0.31%,(flux-weighted)
Median Rel. Diff,1.16%,(robust to outliers)
Agreement Quality,Excellent,(144/175 bins analyzed)


#### Spectral Agreement (YAMC gpu vs OpenMC)

Metric,Value,Note
Reduced Chi-Squared,1770.43,"(ideal ~ 1.0, dof=161)"
Weighted Avg Rel. Diff,63.31%,(flux-weighted)
Median Rel. Diff,10.92%,(robust to outliers)
Agreement Quality,Poor,(143/175 bins analyzed)


... (6 outputs left out)

#### Spectral Agreement (YAMC gpu vs OpenMC)

Metric,Value,Note
Reduced Chi-Squared,1773.18,"(ideal ~ 1.0, dof=160)"
Weighted Avg Rel. Diff,83.04%,(flux-weighted)
Median Rel. Diff,9.58%,(robust to outliers)
Agreement Quality,Poor,(144/175 bins analyzed)


#### Tally Comparison

MT,Reaction,OpenMC (cpu),YAMC (cpu),YAMC (gpu),Rel. Diff cpu (%),Rel. Diff gpu (%)
1,"(n,total)",8.2847e-01 +/- 3.28e-04,8.2891e-01 +/- 3.06e-04,8.8418e-01 +/- 3.40e-04,0.05%,6.73%
2,"(n,elastic)",4.4640e-01 +/- 2.28e-04,4.4665e-01 +/- 1.98e-04,4.7679e-01 +/- 2.11e-04,0.06%,6.81%
16,"(n,2n)",1.8715e-01 +/- 1.87e-05,1.8712e-01 +/- 1.74e-05,2.0114e-01 +/- 2.60e-05,-0.02%,7.48%
17,"(n,3n)",2.4097e-02 +/- 3.03e-06,2.4094e-02 +/- 2.75e-06,2.4920e-02 +/- 2.34e-06,-0.01%,3.42%
18,"(n,fission)",1.2225e-02 +/- 8.32e-07,1.2226e-02 +/- 6.17e-07,1.3043e-02 +/- 2.15e-06,0.01%,6.69%
27,"(n,absorption)",2.7051e-02 +/- 2.37e-05,2.7059e-02 +/- 1.48e-05,2.9349e-02 +/- 1.39e-05,0.03%,8.49%
37,"(n,4n)",2.4260e-13 +/- 1.60e-13,2.3051e-13 +/- 1.80e-13,0.0000e+00 +/- 0.00e+00,-4.98%,-100.00%
51,"(n,n1)",3.9697e-03 +/- 5.90e-06,3.9763e-03 +/- 4.37e-06,4.2475e-03 +/- 5.08e-06,0.17%,7.00%
52,"(n,n2)",1.7084e-02 +/- 9.71e-06,1.7102e-02 +/- 9.61e-06,1.8187e-02 +/- 1.08e-05,0.11%,6.46%
53,"(n,n3)",2.1944e-04 +/- 2.49e-07,2.2002e-04 +/- 1.83e-07,2.3312e-04 +/- 2.95e-07,0.27%,6.24%
